# Características RoCoLe — inventario con máscara COCO

Objetivo: armar un inventario de las mismas 9 medidas de color y textura que el cuaderno 02, pero midiendo solo dentro de la máscara oficial Leaf de COCO, y guardar un CSV paralelo. No hay EDA largo ni modelos: es un inventario masked autocontenido.

Responsables: Víctor y Gio.

## Llegar al índice, al COCO y fijar la semilla

Necesitamos la carpeta featured, el índice partido, el JSON oficial de anotaciones y una carpeta de figuras. Todavía no abrimos fotos.

In [1]:
# Entrada
from pathlib import Path

# Qué hacemos
# semilla fija para muestreos reproducibles
seed = 42
repo_root = Path("../..").resolve()
featured_dir = repo_root / "data" / "featured" / "rocole"
index_path = featured_dir / "index.csv"
coco_json_path = (
    repo_root / "data" / "raw" / "RoCoLe" / "Annotations" / "RoCoLE-coco.json"
)
results_dir = repo_root / "experiments" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

# Salida
print("repo_root:", repo_root)
print("index_path:", index_path)
print("existe índice:", index_path.exists())
print("coco_json_path:", coco_json_path)
print("existe coco:", coco_json_path.exists())
print("results_dir:", results_dir)
seed

repo_root: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe
index_path: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\data\featured\rocole\index.csv
existe índice: True
coco_json_path: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\data\raw\RoCoLe\Annotations\RoCoLE-coco.json
existe coco: True
results_dir: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\experiments\results


42

## Cargar el índice partido

El índice es la mesa de trabajo: una fila por foto, con etiqueta, partición y ruta. No rehacemos el split.

In [2]:
# Entrada
import pandas as pd

# Qué hacemos
rocole_index = pd.read_csv(index_path)

# Salida
print("filas:", len(rocole_index))
print("columnas:", list(rocole_index.columns))
rocole_index.head()

filas: 1481
columnas: ['File', 'plant_key', 'link_group', 'Binary.Label', 'Multiclass.Label', 'leaf_face', 'split', 'photo_path']


,File,plant_key,link_group,Binary.Label,Multiclass.Label,leaf_face,split,photo_path
0,C1P1H1.jpg,C1P1,74,healthy,healthy,H,train,data/curated/rocole/photos/C1P1H1.jpg
1,C1P2E2.jpg,C1P2,84,unhealthy,rust_level_2,E,train,data/curated/rocole/photos/C1P2E2.jpg
2,C1P2H1.jpg,C1P2,84,healthy,healthy,H,train,data/curated/rocole/photos/C1P2H1.jpg
3,C1P3E1.jpg,C1P3,95,healthy,healthy,E,train,data/curated/rocole/photos/C1P3E1.jpg
4,C1P3E2.jpg,C1P3,95,unhealthy,rust_level_2,E,train,data/curated/rocole/photos/C1P3E2.jpg


## Columnas de identidad que viajan con cada fila

Además de las medidas, cada fila del inventario lleva archivo, split, grupo de enlace y las dos etiquetas del índice.

In [3]:
# Entrada
# rocole_index: mesa con File, split, link_group y etiquetas

# Qué hacemos
print("filas:", len(rocole_index))
print("splits:", rocole_index["split"].value_counts().to_dict())
id_columns = [
    "File",
    "split",
    "link_group",
    "Binary.Label",
    "Multiclass.Label",
]

# Salida
print("id_columns:", id_columns)
rocole_index[id_columns].head()

filas: 1481
splits: {'train': 1037, 'test': 226, 'val': 218}
id_columns: ['File', 'split', 'link_group', 'Binary.Label', 'Multiclass.Label']


,File,split,link_group,Binary.Label,Multiclass.Label
0,C1P1H1.jpg,train,74,healthy,healthy
1,C1P2E2.jpg,train,84,unhealthy,rust_level_2
2,C1P2H1.jpg,train,84,healthy,healthy
3,C1P3E1.jpg,train,95,healthy,healthy
4,C1P3E2.jpg,train,95,unhealthy,rust_level_2


## Cargar las anotaciones COCO

Necesitamos el JSON oficial de segmentación para saber qué píxeles son hoja en cada foto.

In [4]:
# Entrada
# si falta: pip install pycocotools
from pycocotools.coco import COCO

# Qué hacemos
coco = COCO(str(coco_json_path))

# Salida
print("coco_json_path:", coco_json_path)
print("imágenes en COCO:", len(coco.getImgIds()))
coco

loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
coco_json_path: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\data\raw\RoCoLe\Annotations\RoCoLE-coco.json
imágenes en COCO: 1560


## Constantes de medición

Fijamos el eps de los índices, los 32 niveles de gris del GLCM y la lista de las nueve columnas de features que debe devolver el inventario.

In [5]:
# Entrada
# ninguna nueva; constantes del protocolo masked

# Qué hacemos
eps = 1e-8
gray_level_count = 32
feature_columns = [
    "exg_mean",
    "exg_std",
    "ngrdi_mean",
    "ngrdi_std",
    "vari_mean",
    "vari_std",
    "glcm_contrast",
    "glcm_homogeneity",
    "glcm_energy",
]

# Salida
print("eps:", eps)
print("gray_level_count:", gray_level_count)
feature_columns

eps: 1e-08
gray_level_count: 32


['exg_mean',
 'exg_std',
 'ngrdi_mean',
 'ngrdi_std',
 'vari_mean',
 'vari_std',
 'glcm_contrast',
 'glcm_homogeneity',
 'glcm_energy']

## Cargar la máscara COCO por File

En el JSON, file_name es una URL; emparejamos si termina en el File del índice. Devolvemos la unión (OR) de las anotaciones Leaf.

In [6]:
# Entrada
# coco: anotaciones RoCoLe ya cargadas

# Qué hacemos
def load_coco_leaf_mask(coco, file_name, height, width):
    """Máscara Leaf (OR) emparejando file_name del JSON por sufijo File."""
    matched_images = [
        img
        for img in coco.loadImgs(coco.getImgIds())
        if img["file_name"].endswith(file_name)
    ]
    if len(matched_images) != 1:
        raise ValueError(
            f"esperaba 1 imagen COCO para {file_name}, hay {len(matched_images)}"
        )

    coco_image = matched_images[0]
    # categoría Leaf = 1; imgIds en lista (ids Labelbox son strings)
    ann_ids = coco.getAnnIds(imgIds=[coco_image["id"]], catIds=[1])
    if len(ann_ids) < 1:
        # respaldo por nombre de categoría si catIds=[1] no encuentra
        leaf_cat_ids = [
            cat["id"]
            for cat in coco.loadCats(coco.getCatIds())
            if cat.get("name", "").lower() == "leaf"
        ]
        ann_ids = coco.getAnnIds(imgIds=[coco_image["id"]], catIds=leaf_cat_ids)

    annotations = coco.loadAnns(ann_ids)
    if len(annotations) < 1:
        raise ValueError(f"sin anotaciones Leaf para {file_name}")

    leaf_mask = None
    for annotation in annotations:
        mask_i = coco.annToMask(annotation).astype(bool)
        leaf_mask = mask_i if leaf_mask is None else (leaf_mask | mask_i)

    if leaf_mask.shape != (height, width):
        raise ValueError(
            f"máscara {leaf_mask.shape} vs imagen {(height, width)} para {file_name}"
        )
    return leaf_mask


# Salida
load_coco_leaf_mask

<function __main__.load_coco_leaf_mask(coco, file_name, height, width)>

## Resumir color solo dentro de la hoja

Media y desviación de ExG, NGRDI y VARI únicamente en los píxeles de la máscara. Si la máscara está vacía, no hay números válidos: error explícito.

In [7]:
# Entrada
import numpy as np
# eps ya fijado arriba

# Qué hacemos
def extract_rgb_summary(image, leaf_mask):
    """Media y std de ExG, NGRDI y VARI solo en píxeles de la máscara."""
    if hasattr(image, "convert"):
        image_rgb = np.asarray(image.convert("RGB"), dtype=np.float64)
    else:
        image_rgb = np.asarray(image, dtype=np.float64)

    leaf_mask = np.asarray(leaf_mask, dtype=bool)
    if leaf_mask.sum() == 0:
        raise ValueError("máscara vacía: no hay píxeles de hoja para RGB")

    red_channel = image_rgb[:, :, 0]
    green_channel = image_rgb[:, :, 1]
    blue_channel = image_rgb[:, :, 2]

    channel_sum = red_channel + green_channel + blue_channel + eps
    r = red_channel / channel_sum
    g = green_channel / channel_sum
    b = blue_channel / channel_sum
    exg_map = 2.0 * g - r - b
    ngrdi_map = (green_channel - red_channel) / (green_channel + red_channel + eps)
    vari_map = (green_channel - red_channel) / (
        green_channel + red_channel - blue_channel + eps
    )

    return {
        "exg_mean": float(exg_map[leaf_mask].mean()),
        "exg_std": float(exg_map[leaf_mask].std()),
        "ngrdi_mean": float(ngrdi_map[leaf_mask].mean()),
        "ngrdi_std": float(ngrdi_map[leaf_mask].std()),
        "vari_mean": float(vari_map[leaf_mask].mean()),
        "vari_std": float(vari_map[leaf_mask].std()),
    }


# Salida
extract_rgb_summary

<function __main__.extract_rgb_summary(image, leaf_mask)>

## Resumir textura GLCM solo en la hoja

Recortamos al bounding box de la máscara, ponemos en 0 lo que no es hoja y calculamos contraste, homogeneidad y energía con distancia 1, ángulo 0 y 32 niveles.

In [8]:
# Entrada
from skimage.feature import graycomatrix, graycoprops
# gray_level_count ya fijado arriba

# Qué hacemos
def extract_glcm_summary(image, leaf_mask):
    """GLCM d=1, ángulo 0, 32 niveles sobre bbox con fuera-hoja en 0."""
    if hasattr(image, "convert"):
        gray = np.asarray(image.convert("L"), dtype=np.uint8)
    else:
        gray = np.asarray(image, dtype=np.uint8)

    leaf_mask = np.asarray(leaf_mask, dtype=bool)
    if leaf_mask.sum() == 0:
        raise ValueError("máscara vacía: no hay píxeles de hoja para GLCM")

    rows, cols = np.where(leaf_mask)
    y0, y1 = int(rows.min()), int(rows.max()) + 1
    x0, x1 = int(cols.min()), int(cols.max()) + 1
    work_gray = gray[y0:y1, x0:x1].copy()
    crop_mask = leaf_mask[y0:y1, x0:x1]
    # fuera de hoja dentro del bbox → 0 (constante)
    work_gray[~crop_mask] = 0

    quantized_gray = (
        work_gray.astype(np.float64) / 255.0 * (gray_level_count - 1)
    ).astype(np.uint8)

    glcm = graycomatrix(
        quantized_gray,
        distances=[1],
        angles=[0],
        levels=gray_level_count,
    )
    return {
        "glcm_contrast": float(graycoprops(glcm, "contrast")[0, 0]),
        "glcm_homogeneity": float(graycoprops(glcm, "homogeneity")[0, 0]),
        "glcm_energy": float(graycoprops(glcm, "energy")[0, 0]),
    }


# Salida
extract_glcm_summary

<function __main__.extract_glcm_summary(image, leaf_mask)>

## Filas de smoke (train)

Para probar el camino sin recorrer todo el índice, tomamos seis fotos de aprender: tres sanas y tres afectadas, con la semilla fija.

In [9]:
# Entrada
# rocole_index, seed, id_columns

# Qué hacemos
train_rows = rocole_index.loc[rocole_index["split"] == "train"].reset_index(drop=True)
healthy_train_sample = (
    train_rows.loc[train_rows["Binary.Label"] == "healthy"]
    .sample(n=3, random_state=seed)
    .reset_index(drop=True)
)
unhealthy_train_sample = (
    train_rows.loc[train_rows["Binary.Label"] == "unhealthy"]
    .sample(n=3, random_state=seed)
    .reset_index(drop=True)
)
smoke_train_rows = pd.concat(
    [healthy_train_sample, unhealthy_train_sample],
    ignore_index=True,
)

# Salida
print("smoke_train_rows:", len(smoke_train_rows))
print(smoke_train_rows["Binary.Label"].value_counts().to_string())
smoke_train_rows[id_columns]

smoke_train_rows: 6
Binary.Label
healthy      3
unhealthy    3


,File,split,link_group,Binary.Label,Multiclass.Label
0,C4P39H1.jpg,train,190,healthy,healthy
1,C9P20E2.jpg,train,333,healthy,healthy
2,C1P6H1.jpg,train,105,healthy,healthy
3,C1P26H2.jpg,train,91,unhealthy,rust_level_2
4,C6P40E2.jpg,train,254,unhealthy,red_spider_mite
5,C12P22H2.jpg,train,66,unhealthy,rust_level_1


## Smoke: features con máscara, fila a fila

Sobre esas pocas filas de train leemos la foto, pedimos la máscara Leaf y medimos RGB+GLCM. Si falla (sin máscara u otro error), NaN en features y mensaje en error. No caemos a foto completa.

In [10]:
# Entrada
from PIL import Image
# smoke_train_rows, coco, repo_root
# load_coco_leaf_mask, extract_rgb_summary, extract_glcm_summary
# feature_columns, id_columns

# Qué hacemos
smoke_feature_rows = []
for _, row in smoke_train_rows.iterrows():
    feature_row = {column_name: row[column_name] for column_name in id_columns}
    feature_row["error"] = None
    for column_name in feature_columns:
        feature_row[column_name] = np.nan

    try:
        photo_path = repo_root / row["photo_path"]
        sample_image = Image.open(photo_path).convert("RGB")
        height, width = sample_image.size[1], sample_image.size[0]
        leaf_mask = load_coco_leaf_mask(coco, row["File"], height, width)
        rgb_summary = extract_rgb_summary(sample_image, leaf_mask)
        glcm_summary = extract_glcm_summary(sample_image, leaf_mask)
        feature_row.update(rgb_summary)
        feature_row.update(glcm_summary)
    except Exception as exc:
        # fallo de máscara o lectura: NaN en features, mensaje en error
        feature_row["error"] = str(exc)

    smoke_feature_rows.append(feature_row)

# Salida
print("smoke_feature_rows:", len(smoke_feature_rows))
print("con error:", sum(1 for r in smoke_feature_rows if r["error"]))
smoke_feature_rows

smoke_feature_rows: 6
con error: 0


[{'File': 'C4P39H1.jpg',
  'split': 'train',
  'link_group': 190,
  'Binary.Label': 'healthy',
  'Multiclass.Label': 'healthy',
  'error': None,
  'exg_mean': 0.8744595304596365,
  'exg_std': 0.19569153140674392,
  'ngrdi_mean': 0.2794048023508362,
  'ngrdi_std': 0.08708880751607716,
  'vari_mean': 0.28566796505150605,
  'vari_std': 0.08371673104565851,
  'glcm_contrast': 0.44470972731581965,
  'glcm_homogeneity': 0.9660850631584,
  'glcm_energy': 0.4864409279311582},
 {'File': 'C9P20E2.jpg',
  'split': 'train',
  'link_group': 333,
  'Binary.Label': 'healthy',
  'Multiclass.Label': 'healthy',
  'error': None,
  'exg_mean': 0.1395306234083355,
  'exg_std': 0.06786396339123582,
  'ngrdi_mean': 0.07407329807979936,
  'ngrdi_std': 0.0494639227213861,
  'vari_mean': 0.12486284912785092,
  'vari_std': 0.08042578287076373,
  'glcm_contrast': 0.6538022756158083,
  'glcm_homogeneity': 0.9478603198316743,
  'glcm_energy': 0.47559226230700413},
 {'File': 'C1P6H1.jpg',
  'split': 'train',
  'link

## Tabla smoke y chequeo de columnas / finitos

Pasamos el smoke a DataFrame y comprobamos que estén todas las columnas pedidas y que, sin error, los números sean finitos.

In [11]:
# Entrada
# smoke_feature_rows, feature_columns, id_columns

# Qué hacemos
smoke_feature_table = pd.DataFrame(smoke_feature_rows)

expected_columns = id_columns + feature_columns + ["error"]
missing_columns = [
    name for name in expected_columns if name not in smoke_feature_table.columns
]
assert missing_columns == [], f"faltan columnas: {missing_columns}"

ok_mask = smoke_feature_table["error"].isna()
finite_ok = True
if ok_mask.any():
    feature_block = smoke_feature_table.loc[ok_mask, feature_columns]
    finite_ok = bool(np.isfinite(feature_block.to_numpy(dtype=float)).all())

# Salida
print("filas:", len(smoke_feature_table))
print("columnas esperadas ok:", missing_columns == [])
print("finitos en filas sin error:", finite_ok)
smoke_feature_table

filas: 6
columnas esperadas ok: True
finitos en filas sin error: True


,File,split,link_group,Binary.Label,Multiclass.Label,error,exg_mean,exg_std,ngrdi_mean,ngrdi_std,vari_mean,vari_std,glcm_contrast,glcm_homogeneity,glcm_energy
0,C4P39H1.jpg,train,190,healthy,healthy,None,0.874460,0.195692,0.279405,0.087089,0.285668,0.083717,0.444710,0.966085,0.486441
1,C9P20E2.jpg,train,333,healthy,healthy,None,0.139531,0.067864,0.074073,0.049464,0.124863,0.080426,0.653802,0.947860,0.475592
2,C1P6H1.jpg,train,105,healthy,healthy,None,0.464751,0.155891,0.174130,0.044048,0.219823,0.044840,0.934725,0.884312,0.397183
3,C1P26H2.jpg,train,91,unhealthy,rust_level_2,None,0.381277,0.136840,0.181763,0.087861,0.253830,0.121603,0.462628,0.905025,0.379794
4,C6P40E2.jpg,train,254,unhealthy,red_spider_mite,None,0.154308,0.050862,0.095024,0.038281,0.166206,0.069049,0.769670,0.906093,0.405603
5,C12P22H2.jpg,train,66,unhealthy,rust_level_1,None,0.709865,0.319958,0.299227,0.115215,0.347977,0.104489,0.660764,0.914883,0.346837


## Barrido completo del índice (con máscara)

Recorremos todas las filas de rocole_index con el mismo protocolo masked. Si no hay máscara Leaf, queda error y NaN: no caemos a foto completa ni miramos el test para elegir nada.

In [12]:
# Entrada
# rocole_index, coco, repo_root
# load_coco_leaf_mask, extract_rgb_summary, extract_glcm_summary
# feature_columns, id_columns

# Qué hacemos
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

inventory_feature_rows = []
for _, row in tqdm(
    rocole_index.iterrows(),
    total=len(rocole_index),
    desc="inventario RGB+GLCM masked",
):
    feature_row = {column_name: row[column_name] for column_name in id_columns}
    feature_row["error"] = None
    for column_name in feature_columns:
        feature_row[column_name] = np.nan

    try:
        photo_path = repo_root / row["photo_path"]
        sample_image = Image.open(photo_path).convert("RGB")
        height, width = sample_image.size[1], sample_image.size[0]
        # sin fallback a full-image: si no hay máscara, Exception → error
        leaf_mask = load_coco_leaf_mask(coco, row["File"], height, width)
        rgb_summary = extract_rgb_summary(sample_image, leaf_mask)
        glcm_summary = extract_glcm_summary(sample_image, leaf_mask)
        feature_row.update(rgb_summary)
        feature_row.update(glcm_summary)
    except Exception as exc:
        feature_row["error"] = str(exc)

    inventory_feature_rows.append(feature_row)

# Salida
print("inventory_feature_rows:", len(inventory_feature_rows))
print(
    "con error:",
    sum(1 for r in inventory_feature_rows if r["error"] is not None),
)

D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


inventario RGB+GLCM masked:   0%|          | 0/1481 [00:00<?, ?it/s]

inventario RGB+GLCM masked:   0%|          | 1/1481 [00:00<05:35,  4.41it/s]

inventario RGB+GLCM masked:   0%|          | 2/1481 [00:00<05:40,  4.34it/s]

inventario RGB+GLCM masked:   0%|          | 3/1481 [00:00<05:51,  4.20it/s]

inventario RGB+GLCM masked:   0%|          | 4/1481 [00:00<05:46,  4.26it/s]

... (1,228 outputs left out)

inventario RGB+GLCM masked: 100%|█████████▉| 1479/1481 [08:30<00:00,  4.42it/s]

inventario RGB+GLCM masked: 100%|██████████| 1481/1481 [08:30<00:00,  5.28it/s]

inventario RGB+GLCM masked: 100%|██████████| 1481/1481 [08:30<00:00,  2.90it/s]

inventory_feature_rows: 1481
con error: 0


## Mesa de features del inventario masked

Convertimos la lista del barrido en un DataFrame: rgb_glcm_features_masked.

In [13]:
# Entrada
# inventory_feature_rows

# Qué hacemos
rgb_glcm_features_masked = pd.DataFrame(inventory_feature_rows)

# Salida
print("filas:", len(rgb_glcm_features_masked))
print("columnas:", list(rgb_glcm_features_masked.columns))
rgb_glcm_features_masked.head()

filas: 1481
columnas: ['File', 'split', 'link_group', 'Binary.Label', 'Multiclass.Label', 'error', 'exg_mean', 'exg_std', 'ngrdi_mean', 'ngrdi_std', 'vari_mean', 'vari_std', 'glcm_contrast', 'glcm_homogeneity', 'glcm_energy']


,File,split,link_group,Binary.Label,Multiclass.Label,error,exg_mean,exg_std,ngrdi_mean,ngrdi_std,vari_mean,vari_std,glcm_contrast,glcm_homogeneity,glcm_energy
0,C1P1H1.jpg,train,74,healthy,healthy,None,0.466084,0.197243,0.250576,0.079053,0.348592,0.076514,0.684317,0.895943,0.381573
1,C1P2E2.jpg,train,84,unhealthy,rust_level_2,None,0.324993,0.094060,0.130550,0.068199,0.180603,0.094060,0.636574,0.897060,0.414018
2,C1P2H1.jpg,train,84,healthy,healthy,None,0.427245,0.191211,0.199764,0.056161,0.272501,0.049086,0.650484,0.886744,0.405814
3,C1P3E1.jpg,train,95,healthy,healthy,None,0.222965,0.062609,0.106283,0.021579,0.165645,0.029721,0.549393,0.927267,0.513788
4,C1P3E2.jpg,train,95,unhealthy,rust_level_2,None,0.237410,0.106820,0.062168,0.077000,0.085289,0.112395,0.660769,0.895654,0.415801


## Chequeo de filas del inventario

Comprobamos que haya una fila por foto del índice, que los conteos por split cuadren y cuántas filas trajeron error.

In [14]:
# Entrada
# rgb_glcm_features_masked, rocole_index

# Qué hacemos
inventory_check = {
    "n_index": len(rocole_index),
    "n_features": len(rgb_glcm_features_masked),
    "len_match": len(rgb_glcm_features_masked) == len(rocole_index),
    "split_counts_index": rocole_index["split"].value_counts().to_dict(),
    "split_counts_features": rgb_glcm_features_masked["split"]
    .value_counts()
    .to_dict(),
    "n_errors": int(rgb_glcm_features_masked["error"].notna().sum()),
}

assert inventory_check["len_match"], (
    f"filas features {inventory_check['n_features']} "
    f"!= índice {inventory_check['n_index']}"
)

# Salida
inventory_check

{'n_index': 1481,
 'n_features': 1481,
 'len_match': True,
 'split_counts_index': {'train': 1037, 'test': 226, 'val': 218},
 'split_counts_features': {'train': 1037, 'test': 226, 'val': 218},
 'n_errors': 0}

## Guardar el CSV de features masked

Escribimos rgb_glcm_features_masked.csv en la carpeta featured de RoCoLe. No entrenamos nada aquí.

In [15]:
# Entrada
# rgb_glcm_features_masked, featured_dir

# Qué hacemos
features_csv_path = featured_dir / "rgb_glcm_features_masked.csv"
rgb_glcm_features_masked.to_csv(features_csv_path, index=False)

# Salida
print("guardado:", features_csv_path)
print("existe:", features_csv_path.exists())
features_csv_path

guardado: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\data\featured\rocole\rgb_glcm_features_masked.csv
existe: True


WindowsPath('D:/Maestria/Documentos/Proyecto de innovación tecnologica en IA/Desarrollo/diagnostico-hojas-cafe/data/featured/rocole/rgb_glcm_features_masked.csv')

## Conclusiones

Quedó un CSV paralelo de las mismas nueve medidas RGB+GLCM, pero solo dentro de la máscara oficial Leaf de COCO; si faltaba máscara, la fila quedó con error y NaN, sin caer a foto completa ni entrenar modelos.